# AML_06 — Livrer un système vérifiable

Conception pédagogique : Mathieu Weill.

**QUID : Un autre peut-il retrouver les mêmes prédictions ?**

Atelier guidé : lire, prévoir une sortie, exécuter au signal, expliquer la preuve. Les repères D1 à D12 sont identiques dans les diapositives et le PDF. Aucun envoi Kaggle automatique.

Télécharger le kit complet de la séance ; placer ce notebook dans `MyLittleMLLab/notebooks`, avec `mllab`, les données officielles et l’environnement. Les exemples indiqués comme fictifs sont distincts de la compétition.

[Guide étudiant](../DEMARRER_J6.md) · [PDF étudiant](../lectures/AML_06_cours.pdf)

## D1 — Reprendre la décision de J5

J6 attend runs/J5/decision.json et les mêmes données officielles. Si le fichier manque, terminer J5 ; ne pas inventer une décision gagnante. Le candidat est choisi avant de fabriquer le fichier de prédictions.

**Au signal :** exécuter une cellule, puis nommer une valeur et sa signification. Ne pas enchaîner si l’étape précédente est en erreur.

In [ ]:
from pathlib import Path
import sys, json, os, time, hashlib, platform, subprocess
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
HERE=Path.cwd()
ROOT=next((p if (p/'mllab/competition.py').exists() else p/'student') for p in [HERE,*HERE.parents] if (p/'mllab/competition.py').exists() or (p/'student/mllab/competition.py').exists())
sys.path.insert(0,str(ROOT))
from mllab.experiments_j56 import context,make,fit_budget,tournament,add_blend,summary,freeze,versions
from sklearn.metrics import roc_auc_score
LIMIT=int(os.environ.get('MLLAB_LIMIT','15000'))
ctx=context(ROOT,LIMIT)
print('Train :',len(ctx['X']),'fraudes :',int(ctx['y'].sum()),'| folds : 3 | CPU')
print('Même source et séparation que J3. Contrôle local déjà consulté : pas un test vierge.')
plt.rcParams.update({'figure.figsize':(10,4),'font.size':13})

from mllab.delivery_j56 import build_release,load_release,positive_scores,validate_rows,predict_rows,align_submission,monitoring,create_app
DECISION=ROOT/'runs/J5/decision.json'
if not DECISION.exists():raise FileNotFoundError('Terminer AML_05_atelier.ipynb D10 avant J6 : runs/J5/decision.json manque.')
decision=json.loads(DECISION.read_text());LIMIT=decision['limit']
RUN=ROOT/'runs/J6';RELEASE=RUN/'release';RUN.mkdir(parents=True,exist_ok=True)
print('Choix figé :',decision['selected'],'; refit :',decision['refit_fits_j6'],'fit(s).')

## D2 — Détecter une erreur silencieuse de lignes

Exemple fictif : le sample attend 18, 12, 27 ; les prédictions arrivent dans l’ordre 12, 18, 27. La longueur seule ne détecte pas l’erreur. Aligner par identifiant avant de contrôler le CSV.

**Au signal :** exécuter une cellule, puis nommer une valeur et sa signification. Ne pas enchaîner si l’étape précédente est en erreur.

In [ ]:
sample_demo=pd.DataFrame({'id':[18,12,27],'is_fraud':[0.,0.,0.]})
pred_demo=pd.DataFrame({'id':[12,18,27],'is_fraud':[.9,.1,.3]})
wrong=sample_demo.copy();wrong['is_fraud']=pred_demo.is_fraud.to_numpy()
aligned=align_submission(sample_demo,pred_demo)
display(pd.DataFrame({'id':sample_demo.id,'affectation_silencieuse':wrong.is_fraud,'apres_jointure':aligned.is_fraud}))
assert np.allclose(aligned.is_fraud,[.1,.9,.3])
try:align_submission(sample_demo,pred_demo.iloc[:2])
except ValueError as e:print('Refus attendu :',e)

## D3 — Réentraîner uniquement le candidat choisi

Un refit apprend de nouveaux paramètres sur tout le train de développement de la même taille qu’en J5. Ce n’est pas une nouvelle recherche. Le contrôle local reste séparé ; aucun label du test Kaggle n’est disponible. Le pipeline complet sera sauvegardé.

**Au signal :** exécuter une cellule, puis nommer une valeur et sa signification. Ne pas enchaîner si l’étape précédente est en erreur.

In [ ]:
start=time.perf_counter();ctx,bundle,manifest=build_release(ROOT,DECISION,RELEASE)
print('Refit terminé en',round(time.perf_counter()-start,3),'s ;',manifest['refit_fits'],'fit(s).')
print('Lignes apprises :',manifest['train_rows'])
print('Bundle : modele.joblib + manifest.json')
print('Empreinte modèle :',manifest['artifact_sha256'][:16])

## D4 — Lire le contrat d’entrée

Chaque clé doit être présente ; une valeur null est admise car l’imputation fait partie du pipeline. Les nombres doivent être finis. Les catégories inconnues restent des chaînes et suivent le traitement prévu par le pipeline. Une catégorie admise ne signifie pas une prédiction fiable hors distribution.

**Au signal :** exécuter une cellule, puis nommer une valeur et sa signification. Ne pas enchaîner si l’étape précédente est en erreur.

In [ ]:
contract=bundle['contract'];display(pd.Series(contract))
example=ctx['X'].iloc[:3].copy()
rows=json.loads(example.to_json(orient='records'))
frame=validate_rows(rows,contract)
assert frame.columns.tolist()==contract['columns']
(RELEASE/'exemple_entree.json').write_text(json.dumps({'rows':rows},ensure_ascii=False,indent=2))
expected=positive_scores(bundle,frame)
print('3 lignes valides ; 3 scores :',expected.round(6).tolist())

## D5 — Vérifier les refus avant la démonstration

Modifier une copie : retirer une clé, ajouter une clé, puis placer du texte dans une colonne numérique. Chaque cas doit être refusé. Ne pas réparer silencieusement une entrée mal définie. Tester aussi une valeur null autorisée.

**Au signal :** exécuter une cellule, puis nommer une valeur et sa signification. Ne pas enchaîner si l’étape précédente est en erreur.

In [ ]:
import copy
cases={}
missing=copy.deepcopy(rows);missing[0].pop(contract['columns'][0]);cases['clé absente']=missing
extra=copy.deepcopy(rows);extra[0]['champ_invente']=1;cases['clé supplémentaire']=extra
bad=copy.deepcopy(rows);bad[0][contract['numeric'][0]]='beaucoup';cases['nombre invalide']=bad
for label,batch in cases.items():
 try:validate_rows(batch,contract)
 except ValueError as e:print(label,'→ refus attendu :',e)
 else:raise AssertionError('Le contrat aurait dû refuser : '+label)
nullable=copy.deepcopy(rows);nullable[0][contract['numeric'][0]]=None
assert len(validate_rows(nullable,contract))==3
print('Valeur null : acceptée selon la politique d’imputation déclarée.')

## D6 — Recharger dans un nouveau processus

Le sous-processus n’a ni variables du notebook ni objet déjà entraîné en mémoire. Il lit le bundle et les trois lignes JSON, puis écrit ses résultats. Une empreinte vérifie l’intégrité par rapport au manifeste ; elle ne rend pas fiable un fichier reçu d’un inconnu.

**Au signal :** exécuter une cellule, puis nommer une valeur et sa signification. Ne pas enchaîner si l’étape précédente est en erreur.

In [ ]:
command=[sys.executable,'-m','mllab.delivery_j56','--bundle',str(RELEASE),'--input',str(RELEASE/'exemple_entree.json'),'--output',str(RELEASE/'resultat_recharge.json')]
env=os.environ.copy();env['PYTHONPATH']=str(ROOT)
result=subprocess.run(command,cwd=ROOT,env=env,check=True,text=True,capture_output=True)
print(result.stdout.strip())
reloaded=json.loads((RELEASE/'resultat_recharge.json').read_text())
assert np.allclose(expected,reloaded['scores'],rtol=1e-6,atol=1e-8)
print('Nouveau processus : prédictions identiques dans la tolérance numérique.')
print('Commande pour reproduire :',subprocess.list2cmdline(command))

## D7 — Produire le vrai fichier au bon format

Le modèle figé produit les scores du test officiel. Alignement par ID, colonnes exactes, absence de NaN et scores entre 0 et 1 sont vérifiés. Cette cellule ne soumet rien à Kaggle.

**Au signal :** exécuter une cellule, puis nommer une valeur et sa signification. Ne pas enchaîner si l’étape précédente est en erreur.

In [ ]:
from mllab.competition import xy
batch_test=xy(ctx['test'],'fraud')
predictions=pd.DataFrame({'id':ctx['test'].id.to_numpy(),'is_fraud':positive_scores(bundle,batch_test)})
submission=align_submission(ctx['sample'],predictions)
submission.to_csv(RUN/'submission_J6.csv',index=False)
assert len(submission)==len(ctx['sample']) and submission.id.equals(ctx['sample'].id)
print('CSV local :',len(submission),'lignes ; colonnes :',submission.columns.tolist())
display(submission.head());print('Aucune soumission réalisée. Vérifier ouverture et règles sur Kaggle avant toute démarche distincte.')

## D8 — Tester une API sans dépendre d’un port réseau

FastAPI est l’outil Python qui expose ici le contrat ; HTTP porte les requêtes. Le client de test appelle les mêmes routes dans le processus, sans ouvrir de serveur réseau. L’API reste une démonstration locale et ne constitue pas une production sécurisée.

**Au signal :** exécuter une cellule, puis nommer une valeur et sa signification. Ne pas enchaîner si l’étape précédente est en erreur.

In [ ]:
from fastapi.testclient import TestClient
app=create_app(RELEASE);client=TestClient(app)
health=client.get('/health');assert health.status_code==200
response=client.post('/predictions',json={'rows':rows});assert response.status_code==200
assert np.allclose(response.json()['scores'],expected,rtol=1e-6,atol=1e-8)
invalid=client.post('/predictions',json={'rows':missing});assert invalid.status_code==422
print('GET /health :',health.status_code)
print('POST /predictions valide :',response.status_code,response.json()['scores'])
print('POST /predictions invalide :',invalid.status_code)
# Pour l’affichage dans un navigateur, suivre DEMARRER_J6.md ; ne pas lancer un serveur bloquant dans cette cellule.

## D9 — Observer un changement sans inventer des labels

Démonstration explicitement simulée : augmenter une variable numérique des entrées. On observe des distributions et des scores. Sans les vrais résultats, aucune nouvelle accuracy ni AUC ne peut être calculée.

**Au signal :** exécuter une cellule, puis nommer une valeur et sa signification. Ne pas enchaîner si l’étape précédente est en erreur.

In [ ]:
from mllab.competition import xy
reference=xy(ctx['control'],'fraud').iloc[:500].copy();current=reference.copy()
watch=contract['numeric'][0];current[watch]=current[watch]+reference[watch].std()*2
p_current=positive_scores(bundle,current)
drift,signals=monitoring(reference,current,p_current)
display(drift[drift.variable==watch].round(4));display(pd.Series(signals))
plt.hist(reference[watch].dropna(),bins=25,alpha=.6,label='Référence');plt.hist(current[watch].dropna(),bins=25,alpha=.6,label='Décalage simulé')
plt.xlabel(watch);plt.ylabel('Nombre de lignes');plt.legend();plt.show()
print('Décalage fabriqué pour comprendre ; aucune dérive de production observée.')

## D10 — Définir une réponse à l’alerte

Une alerte de surveillance doit préciser qui vérifie et quoi faire. Le retour arrière recharge une version approuvée ; il n’efface pas les preuves. Ne pas remplacer automatiquement le modèle au premier changement de distribution.

**Au signal :** exécuter une cellule, puis nommer une valeur et sa signification. Ne pas enchaîner si l’étape précédente est en erreur.

In [ ]:
policy={'signal':'Schéma invalide ou changement d’entrée à vérifier','responsable':'Équipe du projet','premiere_action':'Contrôler source, unités et prétraitement','labels_absents':'Ne pas annoncer de perte AUC mesurée','retour_arriere':'Recharger le dernier bundle approuvé après vérification ; conserver le journal','reentrainement':'Nouvelle expérience sous protocole, puis validation humaine'}
(RUN/'surveillance.json').write_text(json.dumps(policy,ensure_ascii=False,indent=2));display(pd.Series(policy))

## D11 — Créer le paquet remis et sa fiche modèle

Le paquet contient le modèle, le manifeste, le code utile, l’exemple d’entrée, les versions et une fiche de limites. Les données officielles complètes et les identifiants ne sont pas inclus. Transmettre le modèle uniquement à une personne autorisée à l’utiliser.

**Au signal :** exécuter une cellule, puis nommer une valeur et sa signification. Ne pas enchaîner si l’étape précédente est en erreur.

In [ ]:
import zipfile
card=f"""# Fiche modèle — MyLittleMLLab
Question : classer le risque de fraude du jeu pédagogique Kaggle.
Candidat : {manifest['selected']}. Lignes de refit : {manifest['train_rows']}.
Sélection : moyenne AUC CV de J5 ; même validation utilisée pour choisir.
Sortie : score de la classe 1, calibration non démontrée.
Limites : données de compétition, contrôle déjà consulté, pas de preuve d’efficacité en entreprise.
Usage : démonstration locale ; pas de décision de blocage automatique.
"""
(RELEASE/'FICHE_MODELE.md').write_text(card)
(RELEASE/'versions.json').write_text(json.dumps(versions(),indent=2))
package=RUN/'livraison_J6.zip'
with zipfile.ZipFile(package,'w',zipfile.ZIP_DEFLATED) as z:
 for name in ['modele.joblib','manifest.json','exemple_entree.json','resultat_recharge.json','FICHE_MODELE.md','versions.json']:z.write(RELEASE/name,'release/'+name)
 for p in (ROOT/'mllab').glob('*.py'):z.write(p,'mllab/'+p.name)
print('Livraison créée :',package.name,'; pas de données officielles brutes dans le paquet.')

## D12 — Défendre une preuve et une limite

Chaque membre explique une étape différente, puis un autre la reformule. La qualité du système comprend les refus corrects et la reproduction. Le passage NLP/vision changera les entrées ; il conservera split, contrat, prétraitement, évaluation et limites.

**Au signal :** exécuter une cellule, puis nommer une valeur et sa signification. Ne pas enchaîner si l’étape précédente est en erreur.

In [ ]:
checks={'decision_figee':DECISION.exists(),'bundle':(RELEASE/'modele.joblib').exists(),'nouveau_processus':(RELEASE/'resultat_recharge.json').exists(),'csv_aligne':submission.id.equals(ctx['sample'].id),'api_valide':response.status_code==200,'api_refus':invalid.status_code==422,'fiche_modele':(RELEASE/'FICHE_MODELE.md').exists(),'surveillance':(RUN/'surveillance.json').exists()}
assert all(checks.values());display(pd.Series(checks,name='preuve_locale'))
(RUN/'bilan.json').write_text(json.dumps({'checks':checks,'score_public':None,'limite':'Ni déploiement production ni preuve de généralisation universelle'},ensure_ascii=False,indent=2))
print('À présenter : entrée → contrat → pipeline → score → fichier/API → preuve → limite.')